# 05 – Demand geography: is supply where the demand is?

## Question
- **Question:** Is Airbnb supply located where the demand is? Which neighbourhoods have more listings than their demand would justify, and which have more demand per listing than the city average?
- **Predictions:** made before looking at the data by a stand-in for the team (an AI agent that had not seen the data). Each prediction is listed in its section and compared with the result.
- **Variables:** `number_of_reviews_ltm` (demand proxy), `estimated_occupancy_l365d` (Inside Airbnb model estimate), number of listings per neighbourhood (supply), `minimum_nights` (rental model from study 04), `neighbourhood_cleansed`, `neighbourhood_group_cleansed`, `latitude`, `longitude`.
- **Methods:** missing values; distribution of the demand proxy (histogram, share of zeros); concentration of demand (Lorenz curve, Gini coefficient, top-10% share); check of the proxy against the occupancy estimate (Spearman correlation) and why that check is circular; supply vs demand per neighbourhood (log-log scatter plot, Spearman correlation, demand index); Kruskal-Wallis test across districts; maps (diverging choropleth, bubble map, heat maps of supply and demand); replication on pairs 2 and 3 with the same functions.

## 1. Load the data

Load the columns this study needs for pair 1 (Barcelona, Los Angeles), the discovery pair. `load_cities` also adds a `city_name` column ordered by pair, and is reused for all six cities in the replication. `CITY_COLORS` fixes one colour per city, so a city has the same colour in every chart.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from src.cities import CITIES, cities_in_pair
from src.load import load_all
from src.maps import add_bubbles, add_choropleth, add_heatmap, base_map, diverging_colormap, save_map

FIG_DIR = Path("../figures")  # PNGs for the report
FIG_DIR.mkdir(exist_ok=True)

# Fixed colour per city (the colour follows the city in every chart)
CITY_COLORS = dict(zip([c.name for c in CITIES.values()],
                       ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]))

cols = [
    "id", "room_type", "neighbourhood_cleansed", "neighbourhood_group_cleansed", "latitude", "longitude",
    "minimum_nights", "availability_365",
    "number_of_reviews", "number_of_reviews_ltm", "reviews_per_month", "first_review", "last_review",
    "estimated_occupancy_l365d",
]


def load_cities(keys: list[str]) -> pd.DataFrame:
    """Load the listings of the given cities, with a `city_name` column ordered like `keys` (by pair)."""
    data = load_all(keys, usecols=cols)
    data["city_name"] = pd.Categorical(data["city"].map({k: c.name for k, c in CITIES.items()}),
                                       categories=[CITIES[k].name for k in keys], ordered=True)
    return data


# Discovery on pair 1 first; the replication at the end uses the same functions for all six cities
df = load_cities(cities_in_pair(1))
df.groupby("city_name").size()

## 2. Missing values and the demand proxy

There is no booking data. The closest proxy for demand is `number_of_reviews_ltm`, the number of reviews in the last 12 months: a guest can leave one review per stay, so reviews count stays (not every guest writes one).

**Why reviews and not another column?** The data has no bookings. `availability_365` cannot separate a night booked by a guest from a night blocked by the host, and `estimated_occupancy_l365d` is computed from the reviews themselves (section 6). A review is the only direct trace of a real stay: a guest can write one only after staying.

**Why the proxy works for comparisons:** not every guest writes a review (Inside Airbnb assumes about half do). The demand index below compares each neighbourhood with the average of its own city. If the share of guests who write a review is similar across a city, it cancels out in that ratio: a neighbourhood with twice as many reviews per listing also has about twice as many stays. The proxy is biased only where the review rate differs between neighbourhoods or types of listing, for example private rooms vs entire homes (section 8b). For the same reason, absolute review counts compare poorly *between* cities, so the cross-city comparison uses shares, Gini coefficients and correlations instead of raw counts.

The table shows the missing values of the demand columns.

**Result:** `number_of_reviews_ltm` and `estimated_occupancy_l365d` are complete. `reviews_per_month`, `first_review` and `last_review` are missing for exactly the same listings (23.2% in Barcelona, 25.7% in Los Angeles): the listings that have never received a review. These values are not unknown but "no review yet", so they would have to be read as 0, not dropped. The study uses `number_of_reviews_ltm`, which needs no such fix.

In [ ]:
# Share of missing values (%) per demand column and city
demand_cols = ["number_of_reviews", "number_of_reviews_ltm", "reviews_per_month",
               "first_review", "last_review", "estimated_occupancy_l365d"]
df[demand_cols].isna().groupby(df["city_name"]).mean().mul(100).round(1).T

## 3. Which listings count: short stays only

Study 04 showed that 37% (Barcelona) and 47% (Los Angeles) of listings require a stay of 30 nights or more. Such a listing can host at most 12 stays a year, so it gets few reviews even when it is fully booked. Counting reviews over all listings would make neighbourhoods with many monthly rentals look like low-demand areas, which is a confounder, not a finding. The demand analysis therefore uses **short-stay listings only** (`minimum_nights <= 6`, the same cut-off as in study 04), for both supply and demand. Section 7 repeats the main correlation with all listings as a robustness check.

In [ ]:
def short_stays(data: pd.DataFrame) -> pd.DataFrame:
    """Short-stay listings: a minimum stay of 1-6 nights (the short-stay group of study 04)."""
    return data[data["minimum_nights"] <= 6]


stays = short_stays(df)
pd.DataFrame({"all_listings": df.groupby("city_name").size(), "short_stay_listings": stays.groupby("city_name").size()})

## 4. Distribution of the demand proxy

How many reviews does a listing get in a year? The distribution is strongly right-skewed, so the table reports the median and upper percentiles rather than the mean. A listing with zero reviews in 12 months is called **dormant**: no stay was recorded for a year. Unlike the inactive listings of study 04, a dormant listing may still have an open calendar. The table covers all listings and short-stay listings; the histogram shows short-stay listings.

**Prediction (stand-in):** about 35% of listings in Barcelona and 40% in Los Angeles have zero reviews in the last 12 months.

**Result:** over all listings, **36.8% (Barcelona) and 44.9% (Los Angeles)** are dormant, close to the prediction. Among short-stay listings the dormant share drops to **23.2% and 22.4%**: a large part of the "dormant" listings are monthly+ listings, which confirms the choice made in section 3. The median short-stay listing got 13 (Barcelona) and 9 (Los Angeles) reviews in 12 months, and 10% of listings got more than 49 and 43. The histogram has a tall bar at zero and a long, flat tail (the bar at 60 collects all listings with 60 or more reviews).

In [ ]:
def demand_summary(data: pd.DataFrame) -> pd.DataFrame:
    """Distribution of reviews in the last 12 months per city: dormant share, median and upper percentiles."""
    g = data.groupby("city_name")["number_of_reviews_ltm"]
    return pd.DataFrame({
        "listings": g.size(),
        "dormant_pct": g.apply(lambda s: (s == 0).mean() * 100),
        "median": g.median(),
        "p75": g.quantile(0.75),
        "p90": g.quantile(0.90),
        "max": g.max(),
    }).round(1)


pd.concat({"all listings": demand_summary(df), "short-stay listings": demand_summary(stays)}, axis=1)

In [ ]:
def plot_reviews_hist(data: pd.DataFrame, filename: str, clip: int = 60) -> None:
    """Histogram of reviews in the last 12 months, one panel per city (% of the city's listings)."""
    city_names = data["city_name"].unique()
    ncols = min(3, len(city_names))
    nrows = -(-len(city_names) // ncols)  # ceiling division
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols + 1, 3.4 * nrows), sharey=True, squeeze=False)
    for ax, city in zip(axes.flat, city_names):
        reviews = data.loc[data["city_name"] == city, "number_of_reviews_ltm"]
        weights = np.full(len(reviews), 100 / len(reviews))  # each listing counts as 100/n %, so bars show % of listings
        ax.hist(reviews.clip(upper=clip), bins=np.arange(0, clip + 2), weights=weights, color=CITY_COLORS[city])
        ax.axvline(reviews.median(), color="black", linewidth=1, linestyle="--")
        ax.text(reviews.median(), 0.9, f" median {reviews.median():.0f}", transform=ax.get_xaxis_transform(), fontsize=9)
        ax.set_title(city)
        ax.set_xlabel(f"Reviews in the last 12 months ({clip} = {clip} or more)")
        ax.grid(axis="y", alpha=0.3)
        ax.set_axisbelow(True)
        ax.spines[["top", "right"]].set_visible(False)
    for ax in axes[:, 0]:
        ax.set_ylabel("% of listings")
    for ax in axes.flat[len(city_names):]:
        ax.set_visible(False)
    fig.suptitle("Demand proxy per short-stay listing")
    plt.tight_layout()
    plt.savefig(FIG_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()


plot_reviews_hist(stays, "study05_reviews_distribution.png")

## 5. Concentration of demand

Is demand spread evenly over the listings or concentrated on a few? The **Lorenz curve** sorts listings from the fewest to the most reviews and plots the cumulative share of reviews against the cumulative share of listings; the diagonal means every listing gets the same. The **Gini coefficient** (0 = perfect equality, 1 = one listing gets everything) and the **share of reviews going to the top 10% of listings** summarise the curve.

**Prediction (stand-in):** the top 10% of short-stay listings in Barcelona get about 40% of the reviews.

**Result:** demand is strongly concentrated: **Gini = 0.61 in both cities**, and the top 10% of short-stay listings receive **38.6% (Barcelona) and 37.6% (Los Angeles)** of all reviews, close to the prediction. Part of this inequality is mechanical: with 23% of listings at zero reviews, the bottom of the Lorenz curve is flat by definition.

In [ ]:
def gini(x) -> float:
    """Gini coefficient of non-negative values (0 = perfect equality, 1 = maximal concentration)."""
    x = np.sort(np.asarray(x, dtype=float))
    cum = np.cumsum(x)
    return (len(x) + 1 - 2 * (cum / cum[-1]).sum()) / len(x)


def top_share(x, top: float = 0.10) -> float:
    """Share (%) of the total that goes to the top `top` fraction of the values."""
    x = np.sort(np.asarray(x, dtype=float))[::-1]
    return x[: int(np.ceil(top * len(x)))].sum() / x.sum() * 100


def concentration(data: pd.DataFrame) -> pd.DataFrame:
    """Gini coefficient and top-10% share of reviews in the last 12 months, per city."""
    g = data.groupby("city_name")["number_of_reviews_ltm"]
    return pd.DataFrame({"gini": g.apply(gini), "top10_pct_of_reviews": g.apply(top_share)}).round(2)


concentration(stays)

In [ ]:
def plot_lorenz(data: pd.DataFrame, filename: str) -> None:
    """Lorenz curves of reviews in the last 12 months, one line per city."""
    fig, ax = plt.subplots(figsize=(6, 5.5))
    for city in data["city_name"].unique():
        x = np.sort(data.loc[data["city_name"] == city, "number_of_reviews_ltm"].to_numpy(dtype=float))
        cum_listings = np.r_[0, np.arange(1, len(x) + 1) / len(x) * 100]
        cum_reviews = np.r_[0, np.cumsum(x) / x.sum() * 100]
        ax.plot(cum_listings, cum_reviews, color=CITY_COLORS[city], linewidth=2, label=f"{city} (Gini {gini(x):.2f})")
    ax.plot([0, 100], [0, 100], color="grey", linestyle="--", linewidth=1, label="Perfect equality")
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 100)
    ax.set_xlabel("Cumulative % of listings (fewest reviews first)")
    ax.set_ylabel("Cumulative % of reviews in the last 12 months")
    ax.set_title("Concentration of demand (short-stay listings)")
    ax.legend(frameon=False, loc="upper left")
    ax.grid(alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    plt.savefig(FIG_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()


plot_lorenz(stays, "study05_lorenz.png")

## 6. Proxy check: reviews vs estimated occupancy

Inside Airbnb also publishes `estimated_occupancy_l365d`, the estimated number of booked nights in the last year. Does it agree with the review count? According to Inside Airbnb's methodology (the "San Francisco model"), the estimate is itself computed from reviews: reviews are converted to bookings with an assumed review rate of 50%, multiplied by an average length of stay (or by the minimum nights, if higher), and the occupancy is capped at 70%. A high correlation is therefore expected **by construction**. It does not validate the proxy; it only shows that both columns carry the same information, so using the estimate instead of the reviews would not add independent evidence. Spearman's rank correlation is used because both variables are heavily skewed.

Source: Inside Airbnb, *Data assumptions*, https://insideairbnb.com/data-assumptions/ (accessed 2026-10-25).

**Result:** Spearman **ρ = 0.997 (Barcelona) and 0.998 (Los Angeles)**, p < 0.001. As the methodology implies, for short-stay listings the occupancy estimate is almost a monotone function of the review count, so it adds no independent information. The study therefore stays with the raw review count, which rests on fewer assumptions.

In [ ]:
def proxy_check(data: pd.DataFrame) -> pd.DataFrame:
    """Spearman correlation between reviews in the last 12 months and the estimated occupancy, per city."""
    rows = {}
    for city, g in data.groupby("city_name"):
        rho, p = stats.spearmanr(g["number_of_reviews_ltm"], g["estimated_occupancy_l365d"])
        rows[city] = {"n": len(g), "spearman_rho": rho, "p_value": p}
    return pd.DataFrame(rows).T.round(3)


proxy_check(stays)

## 7. Supply vs demand per neighbourhood

For each neighbourhood: **supply** = number of short-stay listings, **demand** = the sum of their reviews in the last 12 months. Both are divided by the city total, which gives the neighbourhood's share of the city's supply and of its demand. The **demand index** = demand share / supply share is the number of reviews per listing in the neighbourhood relative to the city average: 1 means demand is proportional to supply, 2 means twice as many reviews per listing as the city average, 0.5 means half. Neighbourhoods with fewer than 20 short-stay listings are left out, because an index computed from a handful of listings is mostly noise.

Spearman's correlation between listings and reviews measures how closely demand follows supply. Because big neighbourhoods have more of both, this correlation is high almost by construction; the more informative number is the correlation between **listings and the demand index**: is demand per listing higher (agglomeration) or lower (saturation) where there are more listings? In the scatter plot every point is one neighbourhood: x = its share of the city's short-stay listings, y = its share of the city's reviews. Points on the dashed diagonal have a demand index of 1. Red points lie above it (more reviews per listing than the city average, index > 1), blue points below it (index < 1), the same colours as on the maps of section 9. Both axes use log scales because neighbourhood sizes span several orders of magnitude (from 0.1% to 16% of a city's listings).

**Prediction (stand-in):** a strong correlation between listings and reviews (about 0.9), mainly because larger neighbourhoods have more of both.

**Result:** demand follows supply closely: Spearman ρ between listings and reviews = **0.97 (Barcelona) and 0.90 (Los Angeles)**, as predicted. With all listings instead of short stays the correlation is lower (0.94 and 0.84), because monthly+ listings add supply without adding reviews. The demand index ranges from 0.25 to 1.69 in Barcelona and from 0.23 to 2.32 in Los Angeles. The more informative result is the correlation between listings and the demand index: **0.47 in Barcelona (p = 0.002)**, so neighbourhoods with more listings also get more reviews per listing (agglomeration: popular areas attract both hosts and guests, and supply has not saturated demand). In Los Angeles it is about zero (−0.03, p = 0.69): how many listings a neighbourhood has says nothing about its demand per listing.

In [ ]:
MIN_LISTINGS = 20  # neighbourhoods with fewer short-stay listings are left out


def by_neighbourhood(data: pd.DataFrame) -> pd.DataFrame:
    """Supply, demand and demand index per neighbourhood (shares are of the whole city, before the size filter)."""
    t = (data.groupby(["city_name", "neighbourhood_cleansed"])
         .agg(district=("neighbourhood_group_cleansed", "first"), listings=("id", "size"),
              reviews=("number_of_reviews_ltm", "sum"), latitude=("latitude", "mean"), longitude=("longitude", "mean"))
         .reset_index())
    totals = t.groupby("city_name")[["listings", "reviews"]].transform("sum")
    t["supply_share"] = t["listings"] / totals["listings"] * 100
    t["demand_share"] = t["reviews"] / totals["reviews"] * 100
    t["demand_index"] = t["demand_share"] / t["supply_share"]
    return t[t["listings"] >= MIN_LISTINGS].reset_index(drop=True)


def supply_demand_correlation(t: pd.DataFrame) -> pd.DataFrame:
    """Spearman correlations per city: listings vs reviews, and listings vs demand index."""
    rows = {}
    for city, g in t.groupby("city_name"):
        rho, p = stats.spearmanr(g["listings"], g["reviews"])
        rho_index, p_index = stats.spearmanr(g["listings"], g["demand_index"])
        rows[city] = {"neighbourhoods": len(g), "rho_listings_vs_reviews": rho, "p_value": p,
                      "rho_listings_vs_index": rho_index, "p_value_index": p_index,
                      "index_min": g["demand_index"].min(), "index_max": g["demand_index"].max()}
    return pd.DataFrame(rows).T.round(3)


nbhd = by_neighbourhood(stays)
pd.concat({"short-stay listings": supply_demand_correlation(nbhd),
           "all listings (robustness)": supply_demand_correlation(by_neighbourhood(df))})

In [ ]:
def plot_supply_demand(t: pd.DataFrame, filename: str) -> None:
    """Log-log scatter of supply share vs demand share per neighbourhood, one panel per city.

    Red: more reviews per listing than the city average (above the diagonal); blue: fewer (below it),
    the same colours as the diverging scale of the maps.
    """
    city_names = t["city_name"].unique()
    ncols = min(3, len(city_names))
    nrows = -(-len(city_names) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols + 0.5, 4.6 * nrows), squeeze=False)
    for ax, city in zip(axes.flat, city_names):
        g = t[t["city_name"] == city]
        above = g["demand_index"] >= 1
        for mask, color, label in [(above, "#c23b3a", "More reviews per listing than the city average (index > 1)"),
                                   (~above, "#1c5cab", "Fewer reviews per listing than the city average (index < 1)")]:
            ax.scatter(g.loc[mask, "supply_share"], g.loc[mask, "demand_share"], s=24, color=color, alpha=0.85,
                       edgecolor="white", linewidth=0.5, label=label)
        lim = [min(g["supply_share"].min(), g["demand_share"].min()) * 0.7,
               max(g["supply_share"].max(), g["demand_share"].max()) * 1.4]
        ax.plot(lim, lim, color="grey", linestyle="--", linewidth=1)  # demand index = 1
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlim(lim)
        ax.set_ylim(lim)
        ax.set_title(city)
        ax.set_xlabel("Share of the city's short-stay listings (%)")
        ax.set_ylabel("Share of the city's reviews (%)")
        ax.spines[["top", "right"]].set_visible(False)
    for ax in axes.flat[len(city_names):]:
        ax.set_visible(False)
    fig.suptitle("Supply vs demand per neighbourhood (dashed line: demand proportional to supply)")
    plt.tight_layout()
    handles, legend_labels = axes.flat[0].get_legend_handles_labels()
    fig.legend(handles, legend_labels, loc="upper center", bbox_to_anchor=(0.5, 0), ncol=2, frameon=False)
    plt.savefig(FIG_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()


plot_supply_demand(nbhd, "study05_supply_vs_demand.png")

## 8. Where is demand per listing highest?

First a test at district level: do reviews per listing differ between the districts (neighbourhood groups) of a city? The **Kruskal-Wallis test** is the non-parametric version of one-way ANOVA: it compares the rank distributions of several groups and does not assume normality, which the skewed review counts violate. With thousands of listings even a tiny difference is significant, so the effect size **ε² = H / (n − 1)** is reported too (roughly 0.01 small, 0.06 medium, 0.14 large). Then the demand index per district, and the five neighbourhoods with the highest and the lowest index per city. In Los Angeles the three "districts" are the City of Los Angeles, the other cities of the county and the unincorporated areas, so the neighbourhood list is more informative there.

**Prediction (stand-in):** in Barcelona, demand per listing is highest in the tourist centre (Ciutat Vella, Eixample). In Los Angeles it is highest on the coastal Westside (Venice, Santa Monica), not downtown.

**Result:** reviews per listing differ significantly between districts (Kruskal-Wallis p < 0.001 in both cities), but the effect is **small in Barcelona (ε² = 0.035) and negligible in Los Angeles (ε² = 0.004)**: with this many listings, "significant" does not mean "large". In Barcelona the highest demand index is in Gràcia (1.17), Sant Martí (1.13), Eixample (1.09) and Sants-Montjuïc (1.06), and the lowest in Nou Barris (0.29), Sant Andreu (0.56) and, surprisingly, **Ciutat Vella (0.69)**, the most touristic district, where the median short-stay listing got only 3 reviews. The prediction is right for Eixample and wrong for Ciutat Vella (see 8b). In Los Angeles the top neighbourhoods (Jefferson Park, Atwater Village, Bell, Del Aire, Reseda) are not on the coast, and Brentwood on the Westside is among the lowest, so "coastal Westside highest" does not hold. The top neighbourhoods have small supply (20–65 listings), so their index is the least certain.

In [ ]:
def district_test(data: pd.DataFrame) -> pd.DataFrame:
    """Kruskal-Wallis test of reviews per listing across districts, per city, with effect size epsilon squared."""
    rows = {}
    for city, g in data.dropna(subset=["neighbourhood_group_cleansed"]).groupby("city_name"):
        groups = [s.to_numpy() for _, s in g.groupby("neighbourhood_group_cleansed")["number_of_reviews_ltm"]]
        h, p = stats.kruskal(*groups)
        rows[city] = {"districts": len(groups), "n": len(g), "H": h, "p_value": p, "epsilon_sq": h / (len(g) - 1)}
    return pd.DataFrame(rows).T.round(3)


district_test(stays)

In [ ]:
def district_index(data: pd.DataFrame) -> pd.DataFrame:
    """Demand index per district: mean reviews per listing in the district / mean in the city."""
    t = (data.assign(city_mean=data.groupby("city_name")["number_of_reviews_ltm"].transform("mean"))
         .groupby(["city_name", "neighbourhood_group_cleansed"])
         .agg(listings=("id", "size"), median_reviews=("number_of_reviews_ltm", "median"),
              mean_reviews=("number_of_reviews_ltm", "mean"), city_mean=("city_mean", "first")))
    t["demand_index"] = t["mean_reviews"] / t["city_mean"]
    return t.drop(columns="city_mean").sort_values(["city_name", "demand_index"], ascending=[True, False]).round(2)


district_index(stays)

### 8b. Composition check: entire homes only

Ciutat Vella has a different mix of listings: 44.5% of its short-stay listings are private rooms (21.1% in the rest of Barcelona), and a private room gets a median of 0–1 reviews a year against about 20 for an entire home. Room type is therefore a possible confounder of the district comparison. The same index, computed for entire homes only:

**Result:** with entire homes only, Ciutat Vella's index rises from 0.69 to **0.80** but stays below the city average, and the order of the other Barcelona districts barely changes (Gràcia 1.15 and Eixample 1.05 at the top, Nou Barris 0.28 at the bottom). In Los Angeles the three districts stay within 0.98–1.04. The room mix explains part of the low demand per listing in Ciutat Vella, not all of it.

In [ ]:
district_index(stays[stays["room_type"] == "Entire home/apt"])

In [ ]:
def extremes(t: pd.DataFrame, n: int = 5) -> pd.DataFrame:
    """The n neighbourhoods with the highest and the n with the lowest demand index, per city."""
    ranked = t.sort_values("demand_index", ascending=False)
    top = ranked.groupby("city_name").head(n).assign(group="highest")
    bottom = ranked.groupby("city_name").tail(n).assign(group="lowest")
    columns = ["city_name", "group", "neighbourhood_cleansed", "district", "listings", "reviews", "demand_index"]
    return pd.concat([top, bottom])[columns].sort_values(["city_name", "group", "demand_index"],
                                                          ascending=[True, True, False]).round(2)


extremes(nbhd)

## 9. Maps

One interactive map per city (saved in `maps/` as HTML) with four layers:
1. **Choropleth** of the demand index per neighbourhood on a **diverging scale**: blue = fewer reviews per listing than the city average, light grey = average, red = more. The scale runs from 0.5 to 2 in every city, so the maps are comparable; it is symmetric in ratios (half the average is as far from the centre as twice the average). Neighbourhoods with fewer than 20 short-stay listings are drawn in a darker grey.
2. **Bubble map:** one circle per neighbourhood at the mean position of its listings; the area shows the number of short-stay listings (supply) and the colour the demand index.
3. **Heat map of supply:** every short-stay listing counts once.
4. **Heat map of demand:** every short-stay listing is weighted by its reviews in the last 12 months.

Layers 3 and 4 side by side show where demand is more concentrated than supply. Interactive features: three base maps (TMS), layer control, custom colour scale, tooltips on hover and popups on click. The functions live in `src/maps.py`. Screenshots for the report: `figures/study05_map_demand_<city>.png`.

**Result:** in Barcelona the above-average (red) neighbourhoods form a belt from Gràcia through the Eixample to Poblenou and Sants, while the old town (Ciutat Vella), the port area and the outer north are below average (blue). The tourist demand per listing is therefore highest just outside the historic centre, not in it.

In [ ]:
INDEX_SCALE = 2  # colour scale from 1/2 to 2 times the city average


def make_maps(data: pd.DataFrame, t: pd.DataFrame, show: bool = True) -> None:
    """One interactive demand map per city (see the description above)."""
    for city in data["city"].unique():
        city_df = data[data["city"] == city]
        city_t = t[t["city_name"] == CITIES[city].name]
        # the same scale in every city, so maps are comparable; indices below 0.5 or above 2 get the end colours
        cmap = diverging_colormap(INDEX_SCALE, caption="Demand index (reviews per listing / city average)")
        m = base_map(city_df)
        add_choropleth(m, city, city_t.set_index("neighbourhood_cleansed")["demand_index"],
                       city_df.groupby("neighbourhood_cleansed").size(), name="Demand index",
                       legend="", fmt="{:.2f}", min_count=MIN_LISTINGS, colormap=cmap)
        add_bubbles(m, city_t, size="listings", value="demand_index", colormap=cmap,
                    name="Bubbles: listings (size) and demand index (colour)", label="neighbourhood_cleansed")
        add_heatmap(m, city_df, name="Heat map: supply (listings)")
        add_heatmap(m, city_df, name="Heat map: demand (reviews)", weight="number_of_reviews_ltm")
        path = save_map(m, f"study05_demand_{city}.html")
        print("Saved", path.name)
        if show:
            display(m)


make_maps(stays, nbhd)

# Replication: pairs 2 and 3

The method is frozen after pair 1: the cells below load all six cities and call **the same functions** with the same thresholds (`minimum_nights <= 6`, at least 20 listings per neighbourhood). Pair 2 (Madrid – New York) and pair 3 (Tokyo – Mexico City) test whether the pair-1 findings hold elsewhere.

**Predictions before running the replication:**
- Stand-in agent: Tokyo has the highest share of dormant listings (low confidence; Mexico City as the alternative).
- Written in monologue mode before running: demand follows supply closely in every city (Spearman between listings and reviews above 0.8), and demand is strongly concentrated everywhere (Gini above 0.5). New York, where 82% of listings are monthly+ (study 04), has a small short-stay segment, so its neighbourhood estimates rest on fewer listings.

## R1. Load all six cities

In [ ]:
df_all = load_cities(list(CITIES))
stays_all = short_stays(df_all)
nbhd_all = by_neighbourhood(stays_all)
pd.DataFrame({"all_listings": df_all.groupby("city_name").size(),
              "short_stay_listings": stays_all.groupby("city_name").size(),
              "neighbourhoods_with_20+": nbhd_all.groupby("city_name").size()})

## R2. Key indicators per city

The main numbers of sections 4–8 for all six cities, ordered by pair.

In [ ]:
summary = pd.concat([
    demand_summary(stays_all)[["listings", "dormant_pct", "median"]],
    concentration(stays_all),
    proxy_check(stays_all)[["spearman_rho"]].rename(columns={"spearman_rho": "rho_reviews_vs_occupancy"}),
    supply_demand_correlation(nbhd_all)[["neighbourhoods", "rho_listings_vs_reviews", "rho_listings_vs_index",
                                         "p_value_index", "index_min", "index_max"]],
], axis=1).reindex([c.name for c in CITIES.values()])
summary.insert(0, "pair", [c.pair for c in CITIES.values()])
summary

## R3. Charts for all six cities

The histogram (section 4), the Lorenz curves (section 5) and the supply-demand scatter (section 7), drawn by the same functions.

In [ ]:
plot_reviews_hist(stays_all, "study05_reviews_distribution_all.png")

In [ ]:
plot_lorenz(stays_all, "study05_lorenz_all.png")

In [ ]:
plot_supply_demand(nbhd_all, "study05_supply_vs_demand_all.png")

## R4. Districts, extreme neighbourhoods and maps

The Kruskal-Wallis test and the district index (section 8) for the four cities with districts, the extreme neighbourhoods for all six, and the maps (section 9), saved in `maps/`.

In [ ]:
district_test(stays_all)

In [ ]:
extremes(nbhd_all)

In [ ]:
make_maps(stays_all, nbhd_all, show=False)

### R4b. An unexpected pattern: neighbourhoods next to airports

The lists of extreme neighbourhoods show a pattern nobody predicted: some of the highest demand indices are next to the main airport: Del Aire (next to LAX) in Los Angeles, South Ozone Park (next to JFK) as the top neighbourhood of New York, three neighbourhoods of the Barajas district in Madrid, and Venustiano Carranza, the borough that contains Mexico City's airport. The cell compares these neighbourhoods with the rest of each city. The lists were made by hand from the map (neighbourhoods bordering the main airport); Barcelona's airport lies outside the city, so Barcelona has none.

A possible mechanism is in the proxy itself: reviews count **stays, not nights**. Travellers between flights stay one night, so a listing near an airport hosts more stays per year and collects more reviews, even at the same number of booked nights. The share of listings with a one-night minimum tests this.

**Result:** the airport neighbourhoods have a clearly higher demand index in four of five cities: **Madrid 1.88, Mexico City 1.61, New York 1.43, Los Angeles 1.25**, but not in Tokyo (1.02, Ota Ku with Haneda airport). In Madrid, Mexico City and Los Angeles the airport listings also more often accept a single night (74% vs 56%, 84% vs 61%, 67% vs 53%), which fits the "short stays produce more reviews" mechanism. In New York they do not (60% vs 71%), so there the effect is not explained by the minimum stay. Caveat: the airport groups are small (118–1,367 listings), and the neighbourhood lists were made by hand, after seeing the pattern, so this is a hypothesis generated by the data, not a test of it.

In [ ]:
# Neighbourhoods bordering the main airport, looked up by hand on the map (city-specific, so not a frozen-method result)
AIRPORT_NEIGHBOURHOODS = {
    "los_angeles": ["Westchester", "El Segundo", "Del Aire"],                       # LAX
    "madrid": ["Aeropuerto", "Casco Histórico de Barajas", "Timón", "Alameda de Osuna", "Corralejos"],  # Barajas
    "new_york": ["South Ozone Park", "Springfield Gardens", "East Elmhurst"],       # JFK, LaGuardia
    "tokyo": ["Ota Ku"],                                                            # Haneda
    "mexico_city": ["Venustiano Carranza"],                                         # AICM
}
is_airport = pd.Series(False, index=stays_all.index)
for city, names in AIRPORT_NEIGHBOURHOODS.items():
    is_airport |= (stays_all["city"] == city) & stays_all["neighbourhood_cleansed"].isin(names)

airport = (stays_all.assign(next_to_airport=is_airport,
                            city_mean=stays_all.groupby("city_name")["number_of_reviews_ltm"].transform("mean"))
           .query("city != 'barcelona'")
           .groupby(["city_name", "next_to_airport"])
           .agg(listings=("id", "size"),
                one_night_min_pct=("minimum_nights", lambda s: (s == 1).mean() * 100),
                mean_reviews=("number_of_reviews_ltm", "mean"), city_mean=("city_mean", "first")))
airport["demand_index"] = airport["mean_reviews"] / airport["city_mean"]
airport.drop(columns="city_mean").round(2)

## R5. Replication result

Each pair-1 finding checked on pairs 2 and 3 with the same code. **Replicated** = holds in both cities of the pair, **Partial** = holds in one of them, **Not replicated** = holds in neither.

| Pair-1 finding (Barcelona, Los Angeles) | Pair 2: Madrid – New York | Pair 3: Tokyo – Mexico City |
|---|---|---|
| About 23% of short-stay listings are dormant (23.2%, 22.4%) | **Replicated:** 23.9%, 24.1% | **Partial:** Mexico City 26.4%, Tokyo only 11.6% |
| Demand is strongly concentrated (Gini 0.61; top 10% of listings get about 38% of reviews) | **Replicated:** Gini 0.63 and 0.62; top 10% 39.0% and 40.4% | **Partial:** Mexico City 0.65 and 42.4%; Tokyo less concentrated (0.52 and 33.4%) |
| The occupancy estimate is almost a function of the reviews (ρ = 0.997, 0.998) | **Replicated:** 0.995, 0.964 | **Replicated:** 0.999, 0.999 |
| Demand follows supply across neighbourhoods (ρ = 0.97, 0.90) | **Replicated:** 0.94, 0.87 | **Replicated:** 0.99, 0.95 |
| Demand per listing vs supply: positive in Barcelona (ρ = 0.47), none in Los Angeles (−0.03) | **Replicated (the same split):** positive in Madrid (0.37, p < 0.001), negative in New York (−0.31, p = 0.02) | Positive in both: Tokyo 0.44 (p = 0.01), Mexico City 0.58 (p = 0.02) |
| Differences between districts are small (ε² = 0.035, 0.004) | **Partial:** Madrid 0.019 (small), New York 0.103 (medium) | n/a (no districts in the data) |
| Stand-in prediction: Tokyo has the most dormant listings | – | **Not confirmed:** Tokyo has the fewest (11.6%), Mexico City the most (26.4%) |

**Interpretation (draft for the team to check):**
- **What is common to all six cities:** demand is concentrated (top 10% of listings get 33–42% of the reviews), about a quarter of short-stay listings is dormant (except Tokyo), and at neighbourhood level demand follows supply. These look like properties of the platform, not of a city.
- **What differs:** the link between supply and demand per listing. In the four non-US cities, neighbourhoods with more listings also get more reviews per listing (agglomeration). In the two US cities the link is absent (Los Angeles) or reversed (New York, where the most supplied Manhattan neighbourhoods, such as SoHo, Little Italy, the Flatiron District and Murray Hill, have the lowest demand index). Two candidate explanations: (1) regulation, linking to study 04: in New York only 18% of listings are short stays, and under Local Law 18 a short stay is legal only with the host present, so many short-stay listings in central Manhattan may not be bookable in practice; (2) city form: Los Angeles is car-oriented and polycentric, so tourist demand is spread out. The data cannot decide between them.
- **Tokyo stands out** with the fewest dormant listings and the least concentrated demand. A possible reason is Japan's registration system for home-sharing (since 2018), which may keep inactive listings off the platform. To verify.
- **Airports (R4b):** neighbourhoods next to the main airport have 1.25–1.88 times the average demand per listing in four of five cities, partly because one-night stays produce more reviews per booked night. This is a warning about the proxy: the demand index measures the **number of stays**, so areas with short stays look like high-demand areas.

## Findings

### Results on pair 1 (discovery: Barcelona, Los Angeles)
- **Proxy and its limits:** demand is measured by reviews in the last 12 months (one review per stay at most). Monthly+ listings are excluded, because they get few reviews even when fully booked (study 04). Inside Airbnb's occupancy estimate is computed from the same reviews (ρ = 0.997), so it is not an independent check.
- **Dormant listings:** 36.8% (Barcelona) and 44.9% (Los Angeles) of all listings got no review in a year, but only 23.2% and 22.4% of short-stay listings.
- **Concentration:** Gini 0.61 in both cities; the top 10% of short-stay listings get 38.6% and 37.6% of the reviews.
- **Supply vs demand:** across neighbourhoods demand follows supply (ρ = 0.97 and 0.90). In Barcelona, neighbourhoods with more listings also have more demand per listing (ρ = 0.47); in Los Angeles there is no such link.
- **Where demand per listing is highest:** in Barcelona in a belt around the historic centre (Gràcia, Eixample, Poblenou, Sants), not in the centre itself: Ciutat Vella has an index of 0.69 (0.80 for entire homes only). District differences are statistically significant but small (ε² = 0.035 in Barcelona, 0.004 in Los Angeles).

### Cross-city comparison (replication on pairs 2 and 3)
- Concentration, the dormant share and the link between supply and demand replicate in almost all cities (R5), with Tokyo as the exception (11.6% dormant, Gini 0.52).
- The sign of the link between supply and demand per listing splits the cities: positive in Barcelona, Madrid, Tokyo and Mexico City, zero or negative in Los Angeles and New York.
- Unexpected finding: neighbourhoods next to airports have a high demand index in four of five cities (R4b), partly an artefact of the proxy (short stays produce more reviews).
- Main conclusion (draft): **Airbnb supply is largely where the demand is**: no city has large neighbourhoods with many listings and no demand. The mismatches are at the margins and are of two kinds: the historic centre of Barcelona and central Manhattan get less demand per listing than the areas around them, and transit locations (airports) get more.

### Limitations
- **No booking data:** reviews are a proxy for stays, not nights, and not every guest writes a review. The review rate may differ between cities and types of guests.
- **Listing age:** a listing created a few months ago has had less time to collect reviews; the data has no creation date for listings without reviews.
- **Snapshot (June–July 2026):** the last 12 months cover one summer peak; seasonal patterns are not visible.
- **Neighbourhood size differs between cities** (16 boroughs in Mexico City, 165 neighbourhoods with 20+ short-stay listings in Los Angeles), so indices are not like for like across cities, and small neighbourhoods are left out.
- **Correlation, not causation:** room type, price, distance to attractions and regulation all vary between neighbourhoods at the same time.